<a href="https://colab.research.google.com/github/marcohuertas/AI-agents-projects/blob/main/augmenting_biomodels_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA AUGMENTATION
This notebook describes code used to augment the Hugging Face dataset **m1969m/biomodels-sbml-odemodels-oncology** to include:
 - the abstract of the paper associated with the model
 - conversion of SBML code into Tellurium/Antimony that is more human readable

## INSTALLING TELLURIUM

Tellurium is used to convert the model in SBML format to Tellurium/Antimony format that is human readable.

COLAB NOTE: If after running the installation Tellurium doesn't seem to work, then restart session and rerun cells.

In [ ]:
!pip install --upgrade --quiet tellurium

In [ ]:
import tellurium as te
from matplotlib import pyplot as plt

In [ ]:
# Quick test of tellurium
r = te.loada('S1 -> S2; k1*S1; k1 = 0.1; S1 = 10')
r.simulate(0, 50, 100)
r.plot()

## INSTALL RELEVANT PYTHON PACKAGES

In [ ]:
!pip install --quiet boto3

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.client import Config
import requests
import xml.etree.ElementTree as ET
from time import sleep
from tqdm import tqdm
from collections import defaultdict


In [ ]:
from datasets import concatenate_datasets

In [ ]:
from huggingface_hub import notebook_login
from datasets import load_dataset
notebook_login()

### LOAD CUSTOM BIOMODELS DATASET

In [ ]:
# Load biomodels dataset
dataset_name = "m1969m/biomodels-sbml-odemodels-oncology"
ds = load_dataset(dataset_name)

In [ ]:
ds

## ADD PUBMED FULL ABSTRACT


In [ ]:
for idx in range(5,10):
  sample = ds['train'][idx]
  print(sample['pubtype'], sample['accession'], sample['idtype'])

In [ ]:
from google.colab import userdata
email_requester = userdata.get('REQ_EMAIL') # From Secrets or type the actual email

def get_pmcids(sample):
  """
  Get PMCID number using either PMID or DOI labels, using PMC ID Converter
  https://pmc.ncbi.nlm.nih.gov/tools/id-converter-api/
  """
  # NCBI asks that you provide your tool name and an email address in the request
  api_url = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"
  tool_name = "biomodels_article_downloader"
  email_requesting = email_requester
  # email_requesting = "xxx@xxx"

  # Get id type for search
  idtype = sample['idtype']
  ids = sample['accession']

  params = {
    "ids": ids,
    "idtype": idtype,
    "format": "json",
    "tool": tool_name,
    "email": email_requesting
  }

  response = requests.get(api_url, params=params)
  if response.status_code==200:
    data = response.json()

    # Extract the PMCID from the response records
    records = data.get("records", [])
    pmcid = "NoPMCID"
    if records and "pmcid" in records[0]:
      pmcid = records[0]["pmcid"]
  else:
    print(response.status_code)
    print("Error retreiving pmcid for accession = {}".format(ids))
    pmcid = "Error"

  # sleep(1) # Pace the requests to prevent 429 response
  return pmcid

In [ ]:
sample = ds['train'][0]
get_pmcids(sample)

In [ ]:
dstest = ds['train'].select(range(5)).map(lambda u: {'pmcid': get_pmcids(u)})

In [ ]:
dstest

There are dulicated 'accession' values in the dataset. Perhaps there are more than one model per paper, e.g. one for each figure used in the curation.

Identify a list of unique values and make the requests avoiding duplication.

In [ ]:
accession_idtype_list = ds.select_columns(['accession', 'idtype'])['train'].to_pandas().drop_duplicates().to_dict(orient='records')

In [ ]:
pmcid_dict = defaultdict(lambda: "NoPMCID")

for idx in tqdm(range(len(accession_idtype_list))):
  sample = accession_idtype_list[idx]
  pmcid = get_pmcids(sample)
  if pmcid=="Error":
    pmcid_dict[sample['accession']]
  else:
    pmcid_dict[sample['accession']] = pmcid

  sleep(1) # pace yourself, wait a second


In [ ]:
# sample_new = [u for u in accession_idtype_list if u['accession']=='22432059'][0]
# sample_new
# pmcid_dict['22432059'] = get_pmcids(sample_new) #'PMC3304570'

In [ ]:
ds = ds.map(lambda u: {'pmcid': pmcid_dict[u['accession']]})

In [ ]:
for idx in range(0, len(ds['train'])):
  accession = ds['train'][idx]['accession']
  idtype = ds['train'][idx]['idtype']
  pmcid = ds['train'][idx]['pmcid']
  print(accession, idtype, pmcid)


In [ ]:
# Get unique PMC ids to retrieve paper Abstracts
pmcids_list = ds.filter(lambda u: u['pmcid']!='NoPMCID').unique('pmcid')['train']

In [ ]:
# # Get unique PMC ids to retrieve paper Abstracts
# ds_withPMCID = ds.filter(lambda u: u['pmcid'] != 'NoPMCID')
# pmcids_list = list(set(ds_withPMCID['train'][:]['pmcid']))
# pmcids_list

### GET PAPERS FROM PUBMED CENTRAL
Retrieve Abstract from papers

In [ ]:
s3_client = boto3.client("s3", config=Config(signature_version=UNSIGNED))

In [ ]:
def get_paper_xml(pmcid):
  bucket_name = 'pmc-oa-opendata'

  # Structure the prefix matching the S3 schema (e.g., 'PMC8059581.1/')
  prefix = f"{pmcid}.1/"

  response = s3_client.list_objects_v2(
      Bucket=bucket_name,
      Prefix=prefix,
      Delimiter='/',
      MaxKeys=100
  )

  output = None
  xmlfile = "{0}/{0}.xml".format(prefix.split("/")[0])
  # Get xml file to extract abstract
  if 'Contents' in response:
    # print('Searching Contents for xmlfile')
    obj_list = [u['Key'] for u in response['Contents']]
    if xmlfile in obj_list:
      # print("xml file found")
      output = xmlfile
  else:
    print(f"No objects found for prefix: {pmcid}")

  return output, response

In [ ]:
def get_paper_abstract(xmlfile):
  bucket_name = 'pmc-oa-opendata'
  try:
    # 3. Stream the object directly from S3
    response = s3_client.get_object(Bucket=bucket_name, Key=xmlfile)

    # 4. Read the stream and decode it as raw text
    raw_data = response['Body'].read().decode('utf-8')

    # Get Abstract
    root = ET.fromstring(raw_data)

    abstract_paragraphs = root.findall(".//abstract//p")
    abstract_str = "\n".join(
        ["".join([u for u in p.itertext()]) for p in abstract_paragraphs]
        )

    return {"abstract": abstract_str, "error": ""}

  except Exception as e:
    print(f"An error occurred: {e}")

    return {"abstract": "NoAbstract", "error": e}


In [ ]:
import json

def get_paper_json(pmcid):
  bucket_name = 'pmc-oa-opendata'
  prefix = f"{pmcid}.1"
  s3_key = "{0}/{0}.json".format(prefix)

  try:
    # 3. Stream the object directly from S3
    response = s3_client.get_object(Bucket=bucket_name, Key=s3_key)

    # 4. Read the stream and decode it as raw text
    raw_data = response['Body'].read().decode('utf-8')

    # Get Json
    json_data = json.loads(raw_data)

    return json_data

  except Exception as e:
    print(f"An error occurred: {e}")

    return None


In [ ]:
abstract_dict = {'pmcid': [], 'abstract': []}
for pmcid in pmcids_list:
  abstract_dict['pmcid'].append(pmcid)
  xmlfile, _ = get_paper_xml(pmcid) # either filename or None
  if xmlfile:
    abstract_text = get_paper_abstract(xmlfile)
    abstract_dict['abstract'].append(abstract_text['abstract'])
  else:
    abstract_dict['abstract'].append("NoAbstract")

abstract_dict

In [ ]:
pmcid_abstract_dict = defaultdict(lambda: "NoAbstract")
for pmcid, abstr in zip(abstract_dict["pmcid"], abstract_dict["abstract"]):
  pmcid_abstract_dict[pmcid] = abstr

# pmcid_abstract_dict = {pmcid: abstr for pmcid, abstr in zip(abstract_dict["pmcid"], abstract_dict["abstract"])}

In [ ]:
ds_with_abstracts = ds.map(lambda u: {'abstract': pmcid_abstract_dict[u['pmcid']]})

In [ ]:
ds_with_abstracts

# ADD ANTIMONY CODE

In [ ]:
sbml_model = ds['train'][0]

In [ ]:
def get_current_antimony(sample):
  """
  Convert SBML model code to Antimony for easy reading
  """
  id = sample['id']
  sbml_model = sample['sbml_string']
  try:
    r = te.loadSBMLModel(sbml_model)
    antimony_str = r.getCurrentAntimony()
  except Exception as e:
    print(f"An error occurred with BioModel id={id} : {e}")
    antimony_str = ""

  return antimony_str

In [ ]:
print(get_current_antimony(sbml_model))

In [ ]:
ds_with_abstracts_antimony = ds_with_abstracts.map(lambda u: {'antimony_string': get_current_antimony(u)})

In [ ]:
ds_with_abstracts_antimony.filter(lambda u: u['antimony_string']=="")

In [ ]:
# Update dataset in HF
dataset_name = "m1969m/biomodels-sbml-odemodels-oncology"

ds_with_abstracts_antimony.push_to_hub(
    dataset_name,
    commit_message="Updating dataset",
    commit_description="Augmented dataset with paper abstract and conversion of SBML to Antimony where possible."
    )

# DONE